# ctxdrift on Google Colab

Runs the context-drift experiments with open-weight models served by **Ollama** on a Colab GPU.

1. `Runtime > Change runtime type > T4 GPU` (or better).
2. Run the cells in order. The sweep and experiment cells take a while with real models; start small.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: switch the runtime to a GPU'

## 1. Get the code
Upload `ctxdrift-baseline.zip` (made on your Mac, see the README) when the file picker opens.

In [ ]:
import os, zipfile
from google.colab import files
uploaded = files.upload()                 # pick ctxdrift-baseline.zip
for name in uploaded:
    zipfile.ZipFile(name).extractall("/content")

In [ ]:
import glob, os
root = os.path.dirname(glob.glob("/content/**/ctxdrift/pipeline.py", recursive=True)[0].replace("/ctxdrift/pipeline.py", "/x"))
os.chdir(root)
print("Working in", root)
!pip -q install -r requirements.txt
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)

## 2. Start Ollama and pull the models
`qwen3:4b` for the multi-agent system and `qwen3:1.7b` for the modifier fit a T4 together. Check
https://ollama.com/library for newer small Qwen / Gemma tags and change the names below.

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests, os
env = dict(os.environ, OLLAMA_NUM_PARALLEL="2", OLLAMA_KEEP_ALIVE="-1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama is up"); break
    except Exception:
        time.sleep(1)
MAS_MODEL, MOD_MODEL = "qwen3:4b", "qwen3:1.7b"
!ollama pull {MAS_MODEL}
!ollama pull {MOD_MODEL}

## 3. Offline tests (no model calls)

In [ ]:
!python -m pytest -q

## 4. Rough prediction with the simulated agent (seconds, no GPU)
Behavioural model with explicit assumptions (`ctxdrift/sim_agent.py`). Use it to pick the grid for the real sweep.

In [ ]:
!python -m s_values_testing.sweep --backend sim --sim-preset small --runs 20 2>&1 | grep -v '^.modifier]'

## 5. Real sweep of the staleness bounds (s, t_max)
Keep the grid small with real models: 2 runs x 4 scenarios x (2 baselines + 6 V settings) x 7 agent calls ≈ 450 calls.

In [ ]:
!python -m s_values_testing.sweep --backend ollama --mas-model {MAS_MODEL} --modifier-model {MOD_MODEL} \
    --runs 2 --s-values 0,1,2 --t-values off,1 2>&1 | grep -v '^.modifier]'

In [ ]:
import json, glob
rec = json.load(open(sorted(glob.glob("s_values_testing/results/*/recommendation.json"))[-1]))["recommended"]
S = rec["s"]; T = -1 if rec["t_max"] is None else rec["t_max"]
print("Using s =", S, " t_max =", T)

## 6. Main experiment: regular vs full re-read vs source of truth V (in-memory repo, modifier in parallel)

In [ ]:
!python -m ctxdrift.run_experiment --backend ollama --mas-model {MAS_MODEL} --modifier-model {MOD_MODEL} \
    --runs 2 --s {S} --t-max {T} 2>&1 | grep -v '^.modifier]'

## 7. Live on GitHub: Simra-F/sample_repository (the site updates as it runs)
Site: https://simra-f.github.io/sample_repository/

1. Create a **fine-grained token** at github.com/settings/tokens limited to `sample_repository`, with read and write on
   **Contents, Issues, Pull requests** and **Workflows** (the repo has a CI workflow).
2. Add it as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar) and allow this notebook to use it.

Each run resets `main` to the seed files, the modifier pushes commits/issues/PRs as contributor personas while the
MAS works, and the MAS's PR is auto-merged and tested at the end.

In [ ]:
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
os.environ["GITHUB_REPO"] = REPO = "Simra-F/sample_repository"

# (a) Watch the modifier alone change the repo and the site for 3 minutes
!python -m github_modifier --level heavy --minutes 3 --interval 20 --reset --backend ollama --model {MOD_MODEL}

# (b) The experiment against the live repo: regular vs full vs V vs V lean, modifier in parallel
!python -m ctxdrift.run_experiment --github {REPO} --backend ollama --mas-model {MAS_MODEL} \
    --modifier-model {MOD_MODEL} --runs 1 --scenarios moderate,heavy --s {S} --t-max {T}

## 8. Graphs and download

In [ ]:
from IPython.display import Image, display
for d in [sorted(glob.glob("runs/*"))[-1], sorted(glob.glob("s_values_testing/results/*"))[-1]]:
    for png in sorted(glob.glob(f"{d}/*.png")):
        print(png); display(Image(png))

In [ ]:
!zip -qr /content/ctxdrift_results.zip runs s_values_testing/results
files.download("/content/ctxdrift_results.zip")